# Prototyping Functions to Ingest Data

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from typing import List
from fredapi import Fred

## EIA Spot Prices

In [2]:
def fetch_eia_series_pet(series_ids=['RBRTE', 'RWTC'], # ID's of series we're pulling (Brent, WTI)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)


    URL_BASE = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide


def fetch_eia_series_gas(series_ids = ["RNGWHHD"], # ID's of series we're pulling (Henry Hub)
                         frequency= 'weekly', # Frequency of the spot prices, either daily, weekly, or monthly.
                         length = 5000): # Timespan that we're pulling, max 5000 weeks
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)
    

    URL_BASE = f"https://api.eia.gov/v2/natural-gas/pri/fut/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide

fetch_eia_series_gas()
    

series,RNGWHHD
period,
1997-01-10,3.79
1997-01-17,4.19
1997-01-24,2.98
1997-01-31,2.91
1997-02-07,2.53
...,...
2026-05-08,2.74
2026-05-15,2.86
2026-05-22,3.11


## FRED Series (DXY, VIX, 10Y - 2Y Spread)

In [3]:
def fetch_fred_series(series_ids=['T10Y2Y','VIXCLS','DTWEXBGS'], #ID's of the series we want to pull, here 10Y-2Y spread, VIX, and DXY
                      frequency = 'W-FRI'  # How frequent we want the samples to be. (day -> "D", weekly, on friday -> "W-FRI", monthly -> "M", yearly -> "Y"
                      ):
    
    load_dotenv()
    FRED_API_KEY = os.getenv('FRED_API_KEY')
    fred = Fred(api_key= FRED_API_KEY)
    series_dict = {}
    for id in series_ids: # Create a dict with key as ID and values as the corresponding series
        series_dict[id] = fred.get_series(series_id=id)

    df = pd.DataFrame(series_dict) # Turn dictionary into a DF
    df.index = pd.to_datetime(df.index) # Convert index dtype to datetime
    df.index.name = 'period' # Convert index name to 'period' to match the EIA information

    df = df.resample(rule=frequency).last() # Resample to keep only the days at the end of the week

    df = df.dropna() # Drop any NaNs

    return df
fetch_fred_series()

,T10Y2Y,VIXCLS,DTWEXBGS
period,,,
2006-01-06,0.02,11.00,100.0241
2006-01-13,0.02,11.23,99.9675
2006-01-20,0.00,14.56,99.9017
2006-01-27,0.01,11.97,99.6433
2006-02-03,-0.05,12.96,100.1180
...,...,...,...
2026-05-08,0.48,17.19,118.0392
2026-05-15,0.50,18.43,119.2825
2026-05-22,0.43,16.70,119.2868


In [4]:
def fetch_eia_stock(series_ids=['WCESTUS1'], # ID's of series we're pulling (Week-end US Crude Inventory)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)

    URL_BASE = f'https://api.eia.gov/v2/petroleum/stoc/wstk/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}'

    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide

fetch_eia_stock().head()

series,WCESTUS1
period,
1982-08-20,338764
1982-08-27,336138
1982-09-24,335586
1982-10-01,334786
1982-10-08,335260


In [5]:
def get_merged_df(eia_spot_pet_df=fetch_eia_series_pet(), eia_spot_gas_df=fetch_eia_series_gas(), eia_stock_df=fetch_eia_stock(),fred_df=fetch_fred_series()):
    df = pd.merge(left=eia_spot_pet_df, right=fred_df,left_index = True, right_index = True, how= 'inner')
    df = pd.merge(left=df, right=eia_stock_df, left_index = True, right_index = True, how = 'inner')
    df = pd.merge(left=df, right= eia_spot_gas_df, left_index = True, right_index = True, how = 'inner' )
    return df

get_merged_df().head()

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD
period,,,,,,,
2006-01-06,61.72,63.39,0.02,11.00,100.0241,302584,9.42
2006-01-13,62.18,63.74,0.02,11.23,99.9675,305325,8.63
2006-01-20,63.54,66.79,0.00,14.56,99.9017,303016,8.67
2006-01-27,63.77,66.82,0.01,11.97,99.6433,304935,8.22
2006-02-03,64.00,66.59,-0.05,12.96,100.1180,304523,8.36


In [6]:
def compute_returns(df=get_merged_df(), method= 'log',series_ids= ['RBRTE','RWTC', 'RNGWHHD']):
    df = df.copy()
    for id in series_ids:
        df[id + '_log_return'] = np.log(df[id]/ df[id].shift(1))
    return df.dropna()

toy = compute_returns()

    
    


In [7]:
toy.sort_values(by= ['RBRTE_log_return'], ascending= False).head()

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,RBRTE_log_return,RWTC_log_return,RNGWHHD_log_return
period,,,,,,,,,,
2020-05-08,23.57,23.46,0.53,27.98,122.5379,531476,1.84,0.323825,0.400999,0.073272
2009-01-09,45.25,44.46,1.68,42.82,98.8752,308380,5.88,0.200204,0.047442,0.041673
2020-05-22,33.94,33.10,0.49,28.16,122.3211,534422,1.78,0.185982,0.226169,0.088033
2020-04-10,22.53,24.41,0.50,41.67,122.1592,503618,1.77,0.185255,0.118142,0.088553
2020-05-01,17.05,15.71,0.44,37.19,123.1492,532221,1.71,0.180095,1.554333,-0.078692


In [16]:
def rolling_z_scores(df, 
                     series_ids=['RBRTE', 'RWTC', 'RNGWHHD'],
                     window=63,
                     min_periods=1):
    df = df.copy()
    for id in series_ids:
      avg = df[id].rolling(window=window, min_periods=min_periods).mean()
      dev = df[id].rolling(window=window, min_periods=min_periods).std()
      df[id + '_rol_z_score'] = (df[id] - avg) / dev
    return df

rolling_z_scores(get_merged_df())


        


,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,RBRTE_rol_z_score,RWTC_rol_z_score,RNGWHHD_rol_z_score
period,,,,,,,,,,
2006-01-06,61.72,63.39,0.02,11.00,100.0241,302584,9.42,NaN,NaN,NaN
2006-01-13,62.18,63.74,0.02,11.23,99.9675,305325,8.63,0.707107,0.707107,-0.707107
2006-01-20,63.54,66.79,0.00,14.56,99.9017,303016,8.67,1.120079,1.149634,-0.531824
2006-01-27,63.77,66.82,0.01,11.97,99.6433,304935,8.22,0.961228,0.871487,-1.030206
2006-02-03,64.00,66.59,-0.05,12.96,100.1180,304523,8.36,0.936421,0.645228,-0.646171
...,...,...,...,...,...,...,...,...,...,...
2026-05-08,105.88,102.28,0.48,17.19,118.0392,452876,2.74,1.879874,2.544109,-0.497523
2026-05-15,110.53,105.10,0.50,18.43,119.2825,445013,2.86,2.049327,2.565936,-0.420669
2026-05-22,110.61,105.32,0.43,16.70,119.2868,441686,3.11,1.954780,2.414627,-0.270764


In [10]:
def series_diff(df, series_ids=['WCESTUS1']):
    df = df.copy()
    for id in series_ids:
        df[id + '_w_change'] = df[id].diff()
    return df

series_diff(compute_returns(get_merged_df()))

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,RBRTE_log_return,RWTC_log_return,RNGWHHD_log_return,WCESTUS1_w_change
period,,,,,,,,,,,
2006-01-13,62.18,63.74,0.02,11.23,99.9675,305325,8.63,0.007425,0.005506,-0.087591,NaN
2006-01-20,63.54,66.79,0.00,14.56,99.9017,303016,8.67,0.021636,0.046741,0.004624,-2309.0
2006-01-27,63.77,66.82,0.01,11.97,99.6433,304935,8.22,0.003613,0.000449,-0.053299,1919.0
2006-02-03,64.00,66.59,-0.05,12.96,100.1180,304523,8.36,0.003600,-0.003448,0.016888,-412.0
2006-02-10,61.23,63.06,-0.10,12.87,100.3220,309376,7.80,-0.044246,-0.054468,-0.069335,4853.0
...,...,...,...,...,...,...,...,...,...,...,...
2026-05-08,105.88,102.28,0.48,17.19,118.0392,452876,2.74,-0.122097,-0.031660,0.029632,-4306.0
2026-05-15,110.53,105.10,0.50,18.43,119.2825,445013,2.86,0.042981,0.027198,0.042864,-7863.0
2026-05-22,110.61,105.32,0.43,16.70,119.2868,441686,3.11,0.000724,0.002091,0.083801,-3327.0


In [11]:
def lagged_features(df, series_ids= ['RBRTE','RWTC'], lags=[1,4,12]):
    df = df.copy()
    for id in series_ids:
        for lag in lags:
            df[id + f'{lag}_w_lag'] = df.shift(lag)
    return df

In [12]:
def rolling_vol(df, series_ids=['RBRTE_log_return','RWTC_log_return'], periods=[4,12]):
    df = df.copy()
    for id in series_ids:
        id = id.split("_")[0]
        for period in periods:
            df[id + f'_{period}_rol_vol'] = df[id].rolling(window=period,min_periods=1).std()
    return df
rolling_vol(compute_returns(get_merged_df()))




,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,RBRTE_log_return,RWTC_log_return,RNGWHHD_log_return,RBRTE_4_rol_vol,RBRTE_12_rol_vol,RWTC_4_rol_vol,RWTC_12_rol_vol
period,,,,,,,,,,,,,,
2006-01-13,62.18,63.74,0.02,11.23,99.9675,305325,8.63,0.007425,0.005506,-0.087591,NaN,NaN,NaN,NaN
2006-01-20,63.54,66.79,0.00,14.56,99.9017,303016,8.67,0.021636,0.046741,0.004624,0.961665,0.961665,2.156676,2.156676
2006-01-27,63.77,66.82,0.01,11.97,99.6433,304935,8.22,0.003613,0.000449,-0.053299,0.859321,0.859321,1.769642,1.769642
2006-02-03,64.00,66.59,-0.05,12.96,100.1180,304523,8.36,0.003600,-0.003448,0.016888,0.816879,0.816879,1.500144,1.500144
2006-02-10,61.23,63.06,-0.10,12.87,100.3220,309376,7.80,-0.044246,-0.054468,-0.069335,1.283809,1.191021,1.839502,1.843624
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-08,105.88,102.28,0.48,17.19,118.0392,452876,2.74,-0.122097,-0.031660,0.029632,5.961884,18.684594,5.567172,14.303059
2026-05-15,110.53,105.10,0.50,18.43,119.2825,445013,2.86,0.042981,0.027198,0.042864,5.834601,15.751444,4.674830,12.096976
2026-05-22,110.61,105.32,0.43,16.70,119.2868,441686,3.11,0.000724,0.002091,0.083801,5.753511,11.049915,1.537040,8.231440


In [13]:
def differentials(df, series_ids=['RBRTE', 'RWTC']):
    df = df.copy()
    pairings = [(x,y) for i, x in enumerate(series_ids) for y in series_ids[i+1:]] 
    for pair in pairings:
        id_a , id_b = pair 
        df[f'{id_a}_{id_b}_diff'] = df[id_a] - df[id_b]
    return df



In [14]:
series_ids=['RBRTE', 'RWTC', 'BDO']
combinations = []
for id in series_ids:
    for neg_id in series_ids[::-1]:
        pair = set((id, neg_id))
        if pair not in combinations and len(pair) == 2:
            combinations.append(pair)

print(combinations)


combinations = [(x,y) for i, x in enumerate(series_ids) for y in series_ids[i+1::]] # (x,y), get series id and entry index, then do all pairs of x and everything ahead of it. Produces one less each pass, so there is no wasted compute.

[{'BDO', 'RBRTE'}, {'RWTC', 'RBRTE'}, {'RWTC', 'BDO'}]


In [15]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import cloudscraper

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
}

scraper = cloudscraper.create_scraper(
    interpreter='nodejs',  # Use Node.js instead of native solvers for tougher challenges
    browser={
        'browser': 'chrome',
        'platform': 'linux',
        'desktop': True
    })

html = scraper.get(url='https://www.forexfactory.com/calendar/437-opec-meetings')
html_content = html.text
html_content

soup = BeautifulSoup(html_content,'html.parser')


calendar_event = soup.find_all('div', class_= 'flexBox noflex calendar-event-history')
calendar_event


[<div class="flexBox noflex calendar-event-history" data-ebase-id="437"> <div class="head"> <ul> <li class="left noborder nolink"> <strong>History</strong> </li> </ul> </div> <table class="calendar-event__history alternating calendarhistory"> <thead class="subhead"> <tr> <th class="calendarhistory__header calendarhistory__header--history">Expected Impact / Date</th> <th class="calendarhistory__header calendarhistory__header--history">Description</th> </tr> </thead> <tbody> <tr> <td class="calendarhistory__row nowrap calendarhistory__row--history"> <span class="icon icon--ff-impact-ora"></span> <a href="/calendar?day=jun7.2026#detail=149593">Jun 7, 2026</a> </td> <td class="calendarhistory__row calendarhistory__row--description"> <div><span class="darktext"></span></div> </td> </tr> <tr> <td class="calendarhistory__row nowrap calendarhistory__row--history"> <span class="icon icon--ff-impact-ora"></span> <a href="/calendar?day=nov30.2025#detail=145530">Nov 30, 2025</a> </td> <td class="c

In [ ]:
df = pd.read_csv('data/meetings.csv', parse_dates= ['date'])
df_expanded = df.copy()
meeting_dates = df.copy()['date']
df_expanded = df.set_index('date').resample('D').ffill().reset_index()
df['opec_meeting'] = 1
df_expanded = df_expanded.merge(right=df, how='left', on ='date').fillna(0)

df_expanded['days_since_meeting'] = df_expanded["date"].apply(
        lambda d: (d - meeting_dates[meeting_dates <= d].max()).days
        if len(meeting_dates[meeting_dates <= d]) > 0 else None
    )

df_expanded

,date,opec_meeting,days_since_meeting
0,2007-03-15,1.0,0
1,2007-03-16,0.0,1
2,2007-03-17,0.0,2
3,2007-03-18,0.0,3
4,2007-03-19,0.0,4
...,...,...,...
7020,2026-06-03,0.0,185
7021,2026-06-04,0.0,186
7022,2026-06-05,0.0,187
7023,2026-06-06,0.0,188


In [69]:
def days_since_opec():
    df = pd.read_csv('data/meetings.csv', parse_dates= ['date'])
    df = df.copy()
    df_expanded = df.copy()
    meeting_dates = df.copy()['date']
    df_expanded = df.set_index('date').resample('D').ffill().reset_index()
    df['opec_meeting'] = 1
    df_expanded = df_expanded.merge(right=df, how='left', on ='date').fillna(0)

    df_expanded['days_since_meeting'] = df_expanded["date"].apply(
            lambda d: (d - meeting_dates[meeting_dates <= d].max()).days
            if len(meeting_dates[meeting_dates <= d]) > 0 else None
        )
    
    return df_expanded


df = get_merged_df().merge(right = days_since_opec(), left_on='period', right_on='date', how='left')

df.head(30)


,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,date,opec_meeting,days_since_meeting
0,61.72,63.39,0.02,11.00,100.0241,302584,9.42,9999,9999.0,9999.0
1,62.18,63.74,0.02,11.23,99.9675,305325,8.63,9999,9999.0,9999.0
2,63.54,66.79,0.00,14.56,99.9017,303016,8.67,9999,9999.0,9999.0
3,63.77,66.82,0.01,11.97,99.6433,304935,8.22,9999,9999.0,9999.0
4,64.00,66.59,-0.05,12.96,100.1180,304523,8.36,9999,9999.0,9999.0
5,61.23,63.06,-0.10,12.87,100.3220,309376,7.80,9999,9999.0,9999.0
6,58.04,59.37,-0.12,12.01,100.3404,310497,7.25,9999,9999.0,9999.0
7,59.39,59.93,-0.16,11.46,100.2594,312135,7.39,9999,9999.0,9999.0
8,61.06,62.27,-0.08,11.96,99.7024,318776,6.71,9999,9999.0,9999.0
9,59.50,60.89,0.02,11.85,100.9566,323612,6.44,9999,9999.0,9999.0


In [73]:
def days_since_opec(df):
    df = df.copy().reset_index().sort_values('period')
    
    opec_dates = pd.read_csv('data/meetings.csv', parse_dates=['date']).sort_values('date')
    
    df = pd.merge_asof(df, opec_dates.assign(last_opec=opec_dates['date']),
                       left_on='period', right_on='date', direction='backward') #merge_asof used for timeseries data where the dates don't match, finds the closest date in the rightmost df and returns that value for all rows matches, here we use backward to get the most recent date, forward would get the soonest.
    
    df['days_since_opec'] = (df['period'] - df['last_opec']).dt.days
    df = df.drop(columns=['date', 'last_opec'])
    
    return df.set_index('period')

df = days_since_opec(get_merged_df())
df

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,days_since_opec
period,,,,,,,,
2006-01-06,61.72,63.39,0.02,11.00,100.0241,302584,9.42,NaN
2006-01-13,62.18,63.74,0.02,11.23,99.9675,305325,8.63,NaN
2006-01-20,63.54,66.79,0.00,14.56,99.9017,303016,8.67,NaN
2006-01-27,63.77,66.82,0.01,11.97,99.6433,304935,8.22,NaN
2006-02-03,64.00,66.59,-0.05,12.96,100.1180,304523,8.36,NaN
...,...,...,...,...,...,...,...,...
2026-05-08,105.88,102.28,0.48,17.19,118.0392,452876,2.74,159.0
2026-05-15,110.53,105.10,0.50,18.43,119.2825,445013,2.86,166.0
2026-05-22,110.61,105.32,0.43,16.70,119.2868,441686,3.11,173.0
